In [20]:
from __future__ import annotations

import argparse
import sys
from dataclasses import dataclass
from pathlib import Path

import jax
import jax.numpy as jnp
from jax import lax
import numpy as np
import optax

print(jax.__version__) # 0.9.2
print(optax.__version__) # 0.2.8
#Everything will need a flax rewrite. Maybe linen if we are conservative with performance loss.

ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)


from dlgn.models.initialization import init_gate_layer
from dlgn.models.network import run_layer
from dlgn.types import LogicFamily
# from dlgn.models.initialization import init_gate_layer
# from dlgn.models.network import run_layer
# from dlgn.types import LogicFamily

0.9.2
0.2.8
/home/jared/DLGN_Only


In [21]:
# ---------------------------------------------------------------------------
# Helpers to avoid repeating forward_logits' 11-arg signature everywhere
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class HeadConfig:
    """Bundles the decoder/head args that forward_logits needs beyond params/wires/x."""
    architecture: str = 'softmax'
    logic_family: str = 'full'
    class_count: int = 10
    sum_tau: float = 1.0
    gumb_tau: float = 1.0
    dirichlet_concentration: float = 1.0


def head_forward(params, wires, x, training, key, cfg: HeadConfig):
    """Thin wrapper around forward_logits that unpacks a HeadConfig."""
    from dlgn.models.network import forward_logits
    return forward_logits(
        params, wires, x, training, key,
        cfg.architecture, cfg.gumb_tau, cfg.dirichlet_concentration,
        cfg.class_count, cfg.sum_tau, cfg.logic_family,
    )


def evaluate(params, wires, x, labels, key, cfg: HeadConfig) -> dict[str, float]:
    """Run soft + hard forward, return loss and accuracy metrics."""
    logits_soft = head_forward(params, wires, x, True, key, cfg)
    logits_hard = head_forward(params, wires, x, False, key, cfg)
    return {
        'soft_loss': float(optax.softmax_cross_entropy_with_integer_labels(logits_soft, labels).mean()),
        'hard_loss': float(optax.softmax_cross_entropy_with_integer_labels(logits_hard, labels).mean()),
        'soft_acc': float((jnp.argmax(logits_soft, -1) == labels).mean()),
        'hard_acc': float((jnp.argmax(logits_hard, -1) == labels).mean()),
    }


def fmt(metrics: dict[str, float]) -> str:
    return ' '.join(f'{k}={v:.4f}' for k, v in metrics.items())

In [22]:
# ---------------------------------------------------------------------------
# Batch conversion
# ---------------------------------------------------------------------------

def to_flat_jax(batch):
    """Loader batch → (flat float32, int32 labels). For the regular path."""
    from dlgn.utils.batching import batch_to_jax
    return batch_to_jax(batch)


def to_image_jax(batch):
    """Loader batch → (NHWC float32, int32 labels). For the conv path."""
    x, y = batch
    x = x.detach().cpu().numpy() if hasattr(x, 'detach') else np.asarray(x)
    y = y.detach().cpu().numpy() if hasattr(y, 'detach') else np.asarray(y)

    if x.ndim == 4 and x.shape[1] in (1, 3):
        x = np.transpose(x, (0, 2, 3, 1))  # NCHW → NHWC
    elif x.ndim == 3:
        x = x[..., None]

    return jnp.asarray(x.astype(np.float32)), jnp.asarray(y.reshape(-1).astype(np.int32))

In [26]:
# ---------------------------------------------------------------------------
# Init: 
# ---------------------------------------------------------------------------

def _init_tree_kernels(key, input_dim, n_kernels, depth, connection_type, logic_family):
    """Initialize n_kernels independent gate trees over input_dim inputs.

    Each kernel is a binary tree of depth `depth`:
        - leaf inputs = 2^depth  (sampled from input_dim via wiring)
        - gates       = 2^depth - 1
        - layers      = depth

    Returns:
        params: params[k][layer_i] = gate logit array
        wires:  wires[k][layer_i]  = (wa, wb) index arrays
    """
    n_leaves = 2 ** depth
    all_logits = []
    all_wires = []

    for k in range(n_kernels):
        key, subkey = jax.random.split(key)
        tree_logits = []
        tree_wires = []
        layer_in = input_dim

        for layer_i in range(depth):
            subkey, layer_key = jax.random.split(subkey)
            layer_out = n_leaves // (2 ** (layer_i + 1))
            logits, wires = init_gate_layer(
                layer_key, layer_in, layer_out,
                connection_type, logic_family,
            )
            tree_logits.append(logits)
            tree_wires.append(wires)
            layer_in = layer_out

        all_logits.append(tree_logits)
        all_wires.append(tree_wires)

    return all_logits, all_wires

def init_conv_gate_layer(
    key,
    in_channels,
    out_channels,
    kernel_size=(3, 3),
    depth=2,
    connection_type='random',
    logic_family='full',
):
    """Initialize a convolutional DLGN layer.

    Each output channel gets a gate tree wired into patches of size
    in_channels * kh * kw.

    Returns:
        params[k][layer_i], wires[k][layer_i]
    """
    kh, kw = kernel_size
    patch_dim = in_channels * kh * kw
    return _init_tree_kernels(key, patch_dim, out_channels, depth,
                              connection_type, logic_family)

In [27]:
# ---------------------------------------------------------------------------
# Run layer: 
# ---------------------------------------------------------------------------

def _run_tree_kernels(
    params, wires, flat, training, key,
    architecture='softmax', gumb_tau=1.0,
    dirichlet_concentration=1.0, logic_family='full',
):
    """Run all kernel trees on flat input vectors.

    Args:
        params: params[k][layer_i] — gate logits
        wires:  wires[k][layer_i]  — (wa, wb) index pairs
        flat:   (N, input_dim) — batch of flat vectors
        training: bool
        key: PRNG key for stochastic decoders.
        architecture: decoder architecture string.
        gumb_tau: Gumbel-softmax temperature.
        dirichlet_concentration: Dirichlet concentration.
        logic_family: 'full' or 'light'.
    Returns:
        (N, n_kernels)
    """
    n_kernels = len(params)
    depth = len(params[0])

    # one key per kernel × layer
    all_keys = jax.random.split(key, n_kernels * depth)

    def apply_tree(patch_flat, k):
        z = patch_flat
        for layer_i in range(depth):
            layer_key = all_keys[k * depth + layer_i]
            z = run_layer(
                params[k][layer_i],
                wires[k][layer_i],
                z, training, layer_key,
                architecture, gumb_tau,
                dirichlet_concentration, logic_family,
            )
        return z.squeeze(-1)

    def apply_all_kernels(patch_flat):
        return jnp.array([apply_tree(patch_flat, k) for k in range(n_kernels)])

    return jax.vmap(apply_all_kernels)(flat)


def run_conv_gate_layer(
    params, wires, x, training, key,
    kernel_size=(3, 3), stride=(1, 1),
    architecture='softmax', gumb_tau=1.0,
    dirichlet_concentration=1.0, logic_family='full',
):
    """Run a convolutional DLGN layer.

    Extracts patches from the image, runs kernel trees at every
    spatial position (weight sharing), returns spatial feature maps.

    Args:
        params, wires: from init_conv_gate_layer.
        x: (B, H, W, C), values in [0, 1].
        training: bool.
        key: PRNG key for stochastic decoders.
        kernel_size: (kh, kw) — must match init.
        stride: (sh, sw).
        architecture: decoder architecture string.
        gumb_tau: Gumbel-softmax temperature.
        dirichlet_concentration: Dirichlet concentration.
        logic_family: 'full' or 'light'.

    Returns:
        (B, out_h, out_w, out_channels)
    """
    B, H, W, C = x.shape
    kh, kw = kernel_size
    sh, sw = stride
    out_channels = len(params)
    patch_dim = C * kh * kw

    out_h = (H - kh) // sh + 1
    out_w = (W - kw) // sw + 1

    patches = lax.conv_general_dilated_patches(
        x, (kh, kw), (sh, sw),
        padding='VALID',
        dimension_numbers=('NHWC', 'HWIO', 'NHWC'),
    )  # (B, out_h, out_w, patch_dim)

    flat = patches.reshape(B * out_h * out_w, patch_dim)
    out = _run_tree_kernels(
        params, wires, flat, training, key,
        architecture, gumb_tau, dirichlet_concentration, logic_family,
    )
    return out.reshape(B, out_h, out_w, out_channels)


def or_pool(x, kernel_size=(2, 2), stride=None):
    """Or-pooling via max over a spatial window.

    For [0,1] activations, max(a,b) is the t-conorm of a ∨ b.
    Stride defaults to kernel_size (non-overlapping).

    Args:
        x: (B, H, W, C).
        kernel_size: (ph, pw).
        stride: defaults to kernel_size.
    Returns:
        (B, out_h, out_w, C)
    """
    ph, pw = kernel_size
    sh, sw = stride if stride is not None else kernel_size
    return lax.reduce_window(
        x,
        init_value=-jnp.inf,
        computation=lax.max,
        window_dimensions=(1, ph, pw, 1),
        window_strides=(1, sh, sw, 1),
        padding='VALID',
    )



def run_conv_pool_layer(
    params, wires, x, training, key,
    conv_kernel_size=(3, 3), stride=(1, 1),
    architecture='softmax', gumb_tau=1.0,
    dirichlet_concentration=1.0, logic_family='full',
    x, pool_kernel_size=(2, 2), stride=(2,2)
):
    B, H, W, C = x.shape
    kh, kw = conv_kernel_size
    sh, sw = stride
    out_channels = len(params)
    patch_dim = C * kh * kw

    out_h = (H - kh) // sh + 1
    out_w = (W - kw) // sw + 1

    patches = lax.conv_general_dilated_patches(
        x, (kh, kw), (sh, sw),
        padding='SAME',
        dimension_numbers=('NHWC', 'HWIO', 'NHWC'),
    )  # (B, out_h, out_w, patch_dim)

    flat = patches.reshape(B * out_h * out_w, patch_dim)
    out = _run_tree_kernels(
        params, wires, flat, training, key,
        architecture, gumb_tau, dirichlet_concentration, logic_family,
    )
    
    ph, pw = pool_kernel_size
    sh, sw = stride
    
    return lax.reduce_window(
        out,
        init_value=-jnp.inf,
        computation=lax.max,
        window_dimensions=(1, ph, pw, 1),
        window_strides=(1, sh, sw, 1),
        padding='VALID',
    )

In [ ]:
# ---------------------------------------------------------------------------
# Regular (flat) DLGN path
# ---------------------------------------------------------------------------

def run_regular_path(args, train_loader, test_loader, class_count):
    from dlgn.data.loaders import cycle_loader
    from dlgn.models.initialization import init_logic_gate_network
    from dlgn.training.optim import create_optimizer
    from dlgn.training.state import TrainState
    from dlgn.training.steps import make_train_step

    cfg = HeadConfig(
        architecture=args.regular_architecture,
        logic_family=args.regular_logic_family,
        class_count=class_count,
        sum_tau=args.sum_tau,
        gumb_tau=args.gumb_tau,
        dirichlet_concentration=args.dirichlet_concentration,
    )

    # --- Init model ---
    sample_x, _ = to_flat_jax(next(iter(train_loader)))
    input_dim = sample_x.shape[-1]

    key = jax.random.PRNGKey(args.seed)
    key, init_key = jax.random.split(key)
    params, wires = init_logic_gate_network(
        input_dim=input_dim,
        num_neurons=args.regular_neurons,
        num_layers=args.regular_layers,
        connections='random',
        key=init_key,
        logic_family=cfg.logic_family,
    )

    # --- Optimizer ---
    opt_config = {
        'learning_rate': args.learning_rate,
        'weight_decay': args.weight_decay,
        'clip_value': args.clip_value,
    }
    tx = create_optimizer(opt_config)
    state = TrainState(params=params, opt_state=tx.init(params), key=key)
    train_step = make_train_step(tx)

    # --- Eval before training ---
    test_x, test_y = to_flat_jax(next(iter(test_loader)))
    eval_key = jax.random.PRNGKey(args.seed + 10_000)
    before = evaluate(state.params, wires, test_x, test_y, eval_key, cfg)

    # --- Train ---
    train_iter = cycle_loader(train_loader)
    for _ in range(args.train_steps):
        batch_x, batch_y = to_flat_jax(next(train_iter))
        state, loss, _ = train_step(
            state, batch_x, batch_y, wires,
            cfg.architecture, cfg.gumb_tau, cfg.dirichlet_concentration,
            cfg.class_count, cfg.sum_tau, cfg.logic_family,
        )

    # --- Eval after training ---
    eval_key = jax.random.PRNGKey(args.seed + 20_000)
    after = evaluate(state.params, wires, test_x, test_y, eval_key, cfg)

    print('\n[regular] flattened DLGN on MNIST')
    print(f'  input_dim={input_dim}  neurons={args.regular_neurons}  layers={args.regular_layers}')
    print(f'  before: {fmt(before)}')
    print(f'  after:  {fmt(after)}')
    print(f'  last_train_loss={float(loss):.4f}')
    return after


In [28]:
# ---------------------------------------------------------------------------
# Conv-Pool: Run one after the other
# ---------------------------------------------------------------------------

def conv_extract(conv_params, conv_wires, images, training, key,
                 kernel_size, stride, pool_size,
                 conv_architecture='softmax', gumb_tau=1.0,
                 dirichlet_concentration=1.0, conv_logic_family='full'):
    """Conv layer → or-pool → feature maps."""
    from dlgn.models.conv import run_conv_gate_layer, or_pool
    x = run_conv_gate_layer(conv_params, conv_wires, images, training,
                            key=key, kernel_size=kernel_size, stride=stride,
                            architecture=conv_architecture,
                            gumb_tau=gumb_tau,
                            dirichlet_concentration=dirichlet_concentration,
                            logic_family=conv_logic_family)
    if pool_size > 1:
        x = or_pool(x, kernel_size=(pool_size, pool_size))
    return x


def conv_forward(params, wires, images, training, key, cfg: HeadConfig,
                 kernel_size, stride, pool_size,
                 conv_architecture='softmax', conv_logic_family='full'):
    """Full conv pipeline: extract features → flatten → head → class logits."""
    key, conv_key = jax.random.split(key)
    feats = conv_extract(params['conv'], wires['conv'], images, training,
                         conv_key, kernel_size, stride, pool_size,
                         conv_architecture=conv_architecture,
                         gumb_tau=cfg.gumb_tau,
                         dirichlet_concentration=cfg.dirichlet_concentration,
                         conv_logic_family=conv_logic_family)
    flat = feats.reshape(feats.shape[0], -1)
    return head_forward(params['head'], wires['head'], flat, training, key, cfg)


def conv_evaluate(params, wires, images, labels, key, cfg, kernel_size, stride, pool_size,
                  conv_architecture='softmax', conv_logic_family='full'):
    """Soft + hard eval for the conv model."""
    key1, key2 = jax.random.split(key)
    logits_soft = conv_forward(params, wires, images, True, key1, cfg,
                               kernel_size, stride, pool_size,
                               conv_architecture, conv_logic_family)
    logits_hard = conv_forward(params, wires, images, False, key2, cfg,
                               kernel_size, stride, pool_size,
                               conv_architecture, conv_logic_family)
    return {
        'soft_loss': float(optax.softmax_cross_entropy_with_integer_labels(logits_soft, labels).mean()),
        'hard_loss': float(optax.softmax_cross_entropy_with_integer_labels(logits_hard, labels).mean()),
        'soft_acc': float((jnp.argmax(logits_soft, -1) == labels).mean()),
        'hard_acc': float((jnp.argmax(logits_hard, -1) == labels).mean()),
    }


def run_conv_path(args, train_loader, test_loader, class_count):
    from dlgn.data.loaders import cycle_loader
    from dlgn.models.conv import init_conv_gate_layer
    from dlgn.models.initialization import init_logic_gate_network
    from dlgn.training.optim import create_optimizer
    from dlgn.training.state import TrainState

    kernel_size = (args.conv_kernel_size, args.conv_kernel_size)
    stride = (args.conv_stride, args.conv_stride)

    # Conv and head can use different logic families/architectures
    conv_logic = args.conv_logic_family
    conv_arch = args.conv_architecture
    head_logic = args.conv_head_logic_family
    head_arch = args.conv_head_architecture

    cfg = HeadConfig(
        architecture=head_arch,
        logic_family=head_logic,
        class_count=class_count,
        sum_tau=args.sum_tau,
        gumb_tau=args.gumb_tau,
        dirichlet_concentration=args.dirichlet_concentration,
    )

    # --- Init conv layer ---
    sample_images, _ = to_image_jax(next(iter(train_loader)))
    in_channels = sample_images.shape[-1]

    key = jax.random.PRNGKey(args.seed + 1_000)
    key, conv_key, head_key, probe_key = jax.random.split(key, 4)
    conv_params, conv_wires = init_conv_gate_layer(
        conv_key,
        in_channels=in_channels,
        out_channels=args.conv_channels,
        kernel_size=kernel_size,
        depth=args.conv_depth,
        connection_type='unique',
        logic_family=conv_logic,
    )

    # --- Init head (sized from a probe forward pass) ---
    probe_feats = conv_extract(conv_params, conv_wires, sample_images, False,
                               probe_key, kernel_size, stride, args.pool_size,
                               conv_architecture=conv_arch,
                               conv_logic_family=conv_logic)
    head_input_dim = int(np.prod(probe_feats.shape[1:]))

    head_params, head_wires = init_logic_gate_network(
        input_dim=head_input_dim,
        num_neurons=args.conv_head_neurons,
        num_layers=args.conv_head_layers,
        connections='random',
        key=head_key,
        logic_family=head_logic,
    )

    params = {'conv': conv_params, 'head': head_params}
    wires = {'conv': conv_wires, 'head': head_wires}

    # --- Optimizer ---
    tx = create_optimizer({
        'learning_rate': args.learning_rate,
        'weight_decay': args.weight_decay,
        'clip_value': args.clip_value,
    })
    state = TrainState(params=params, opt_state=tx.init(params), key=key)

    # --- Eval before training ---
    test_images, test_y = to_image_jax(next(iter(test_loader)))
    eval_key = jax.random.PRNGKey(args.seed + 20_000)
    before = conv_evaluate(state.params, wires, test_images, test_y, eval_key,
                           cfg, kernel_size, stride, args.pool_size,
                           conv_arch, conv_logic)

    # --- Train ---
    train_iter = cycle_loader(train_loader)
    for _ in range(args.train_steps):
        batch_images, batch_y = to_image_jax(next(train_iter))
        key, subkey = jax.random.split(state.key)

        def loss_fn(p):
            logits_s = conv_forward(p, wires, batch_images, True, subkey, cfg,
                                    kernel_size, stride, args.pool_size,
                                    conv_arch, conv_logic)
            logits_h = conv_forward(p, wires, batch_images, False, subkey, cfg,
                                    kernel_size, stride, args.pool_size,
                                    conv_arch, conv_logic)
            soft = optax.softmax_cross_entropy_with_integer_labels(logits_s, batch_y).mean()
            hard = optax.softmax_cross_entropy_with_integer_labels(logits_h, batch_y).mean()
            return soft, {'hard': hard}

        (loss, aux), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
        updates, opt_state = tx.update(grads, state.opt_state, state.params)
        new_params = optax.apply_updates(state.params, updates)
        state = state.replace(params=new_params, opt_state=opt_state, key=key)

    # --- Eval after training ---
    eval_key = jax.random.PRNGKey(args.seed + 30_000)
    after = conv_evaluate(state.params, wires, test_images, test_y, eval_key,
                          cfg, kernel_size, stride, args.pool_size,
                          conv_arch, conv_logic)

    print(f'\n[conv] conv DLGN ({conv_logic}) + head ({head_logic}) on MNIST')
    print(f'  input={tuple(sample_images.shape[1:])}  conv_out={args.conv_channels}  '
          f'depth={args.conv_depth}  pooled={tuple(probe_feats.shape[1:])}  '
          f'head_in={head_input_dim}')
    print(f'  before: {fmt(before)}')
    print(f'  after:  {fmt(after)}')
    print(f'  last_train_loss={float(loss):.4f}')
    return after

In [31]:
config = {
    'dataset': 'mnist', #choices=['mnist', 'mnist20x20', 'mnist_bin', 'mnist20x20_bin'])
    'model': 'both',    #choices=['regular', 'conv', 'both']
    'storage_root': str(ROOT / 'dataset_storage'),
    'batch_size': 128,
    'train_steps': 5000,
    'seed': 0,

    'learning_rate': 0.01,
    'weight_decay': 1e-4,
    'clip_value': 1.0,
    'sum_tau': 1.0,
    'gumb_tau': 1.0,
    'dirichlet_concentration': 1.0,

    'regular_neurons': 1200,
    'regular_layers': 5,

    'regular_logic_family': 'full', #choices=['full', 'light']
    'regular_architecture': 'softmax',

    'conv_channels': 32,
    'conv_depth': 3,
    'conv_kernel_size': 3,
    'conv_stride': 1,
    'conv_head_neurons': 1020,
    'conv_head_layers': 3,

    'conv_logic_family': 'full', #choices=['full', 'light']
    'conv_architecture': 'softmax',

    'pool_size': 2,
    'pool_stride': 2,

}


def main(argv=None) -> int:
    from dlgn.data.loaders import load_dataset
    from dlgn.data.registry import num_classes_of_dataset
    from dlgn.utils.seeding import seed_all


    storage_root = Path(config['storage_root']).resolve()
    seed_all(config['seed'], seed_torch=True)

    data_config = {
        'dataset': config['dataset'],
        'seed': config['seed'],
        'batch_size': config['batch_size'],
        'valid_set_size': 0.0,
        'num_workers': 0,
        'data_roots': {
            'uci': str(storage_root / 'uci'),
            'mnist': str(storage_root / 'mnist'),
            'cifar': str(storage_root / 'cifar'),
            'block': str(storage_root / 'block'),
        },
    }

    train_loader, _, test_loader = load_dataset(data_config)
    class_count = num_classes_of_dataset(config['dataset'])

    print(f'Dataset: {config['dataset']}  classes={class_count}  '
          f'batch={config['batch_size']}  steps={config['train_steps']}')

    if config['model'] in ('regular', 'both'):
        run_regular_path(args, train_loader, test_loader, class_count)

    if config['model'] in ('conv', 'both'):
        run_conv_path(args, train_loader, test_loader, class_count)

    print('\nDone.')
    return 0

In [ ]:
config_small = {
    "dataset": "mnist",
    "model": "conv",

    "batch_size": 512,
    "train_steps": 5000,
    "seed": 0,

    "learning_rate": 0.01,
    "weight_decay": 0.0,
    "clip_value": 1.0,
    "sum_tau": 6.5,

    "conv_block_channels": [16, 48, 144],
    "conv_block_kernel_sizes": [5, 3, 3],
    "conv_block_depths": [3, 3, 3],
    "conv_block_paddings": [0, 1, 1],
    "conv_block_strides": [1, 1, 1],

    "pool_size": 2,
    "pool_stride": 2,

    "fc_sizes": [20480, 10240, 5120],
    "num_classes": 10,

    "logic_family": "full",
    "architecture": "softmax"
}

config_medium = {
    "dataset": "mnist",
    "model": "conv",

    "batch_size": 256,
    "train_steps": 5000,
    "seed": 0,

    "learning_rate": 0.01,
    "weight_decay": 0.0,
    "clip_value": 1.0,
    "sum_tau": 28,

    "conv_block_channels": [64, 192, 576],
    "conv_block_kernel_sizes": [5, 3, 3],
    "conv_block_depths": [3, 3, 3],
    "conv_block_paddings": [0, 1, 1],
    "conv_block_strides": [1, 1, 1],

    "pool_size": 2,
    "pool_stride": 2,

    "fc_sizes": [81920, 40960, 20480],
    "num_classes": 10,

    "logic_family": "full",
    "architecture": "softmax"
}

# FIX THE FC, but I'm not attempting it right now.
# config_large = {
#     "dataset": "mnist",
#     "model": "conv",

#     "batch_size": 128,
#     "train_steps": 5000,
#     "seed": 0,

#     "learning_rate": 0.01,
#     "weight_decay": 0.0,
#     "clip_value": 1.0,
#     "sum_tau": 35,

#     "conv_block_channels": [1024, 3072, 9216],
#     "conv_block_kernel_sizes": [5, 3, 3],
#     "conv_block_depths": [3, 3, 3],
#     "conv_block_paddings": [0, 0, 0],
#     "conv_block_strides": [1, 1, 1],

#     "pool_size": 2,
#     "pool_stride": 2,

#     "fc_sizes": [20480, 10240, 5120],
#     "groupsum_k": 512,
#     "num_classes": 10,

#     "logic_family": "full",
#     "architecture": "softmax"
# }





# (∗): For the S & M size MNIST models, we use 2×as many gates in the final layers.
# • A convolutional block with k kernels with a receptive field of size 5 ×5 and tree depth d= 3, without padding.
# • An or pooling layer with kernel size 2 ×2 and stride 2. [shape after layer: k ×12 ×12]
# • A convolutional block with 3*k kernels with a receptive field of size 3 ×3 and depth d= 3.
# • An or pooling layer with kernel size 2 ×2 and stride 2. [shape after layer: 3*k ×6 ×6]
# • A convolutional block with 9*k kernels with a receptive field of size 3 ×3 and depth d= 3.
# • An or pooling layer with kernel size 2 ×2 and stride 2. [shape after layer: 9*k ×3 ×3]
# • Flattening the hidden state. [shape after flattening: 81*k]
# • Regular differentiable logic layer 81*k →1280*k (∗)
# .
# • Regular differentiable logic layer 1280*k →640*k (∗)
# .
# • Regular differentiable logic layer 640*k →320*k (∗)
# .
# • GroupSum with 10 classes 320*k →10.
# (∗): For the S & M size MNIST models, we use 2×as many gates in the final layers.